In [2]:
import sys
import os
from unsloth import FastVisionModel
import torch
from transformers import TextStreamer
from datasets import load_dataset
from stage_1_datasets_static import build_static_dataset
import json

I0000 00:00:1777659541.657945   14634 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1777659542.280569   14634 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777659544.377993   14634 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [3]:
# ── 1. Load already-downloaded HF datasets (no re-download) ──────────────────
molmoact   = load_dataset("allenai/MolmoAct-Pretraining-Mixture", "auxiliary_trace")["train"]
pixmocap   = load_dataset("allenai/pixmo-cap")["train"]
pixmoama   = load_dataset("allenai/pixmo-ask-model-anything")["train"]
pixmocapqa = load_dataset("allenai/pixmo-cap-qa")["train"]
# ── 2. Build static dataset (runs once, takes a few minutes for RoboVQA scan) ─
#    ShareRobot, RoboFAC, RoboVQA are loaded automatically from data/ folder.
splits = build_static_dataset(molmoact, pixmocap, pixmoama, pixmocapqa)
# splits is a plain dict:  {"train": VLAStaticDataset, "val": ..., "test": ...}
train_ds = splits["train"]
val_ds   = splits["val"]
print(f"Train: {len(train_ds):,}  |  Val: {len(val_ds):,}")


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/165 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/278 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Building static VLA dataset...
  MolmoAct: 195,622 records
  ShareRobot Affordance: 6,522 records
  ShareRobot Planning: 99,985 records
  RoboVQA: 99,988 records (from 801,388 QA pairs)
  pixmocap: 50,000 records
  pixmoama: 49,750 records
  pixmocapqa: 50,000 records
  RoboFAC: 48,993 records

Dataset split summary:
  train:  468,901  (78.0%)
  val  :   78,714  (13.1%)
  test :   53,245  (8.9%)
  total:  600,860
Train: 468,901  |  Val: 78,714


In [4]:
# Import the pretrained base model and the tokenizer from Unsloth
model, tokenizer = FastVisionModel.from_pretrained(
    "unsloth/Qwen3.5-4B",
    load_in_4bit = False,
    use_gradient_checkpointing = "unsloth"
)
print("Imported Base Model")

==((====))==  Unsloth 2026.4.6: Fast Qwen3_5 patching. Transformers: 5.5.4. vLLM: 0.19.1.
   \\   /|    NVIDIA RTX A4000. Num GPUs = 1. Max memory: 15.992 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.6. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

Imported Base Model


In [5]:
# Add LoRA adapters to the model
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers = True,
    finetune_language_layers = True,
    finetune_attention_modules = True,
    finetune_mlp_modules = True,

    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3047,
    use_rslora = False,
    loftq_config = None,
    target_modules = "all-linear",
)
print("Added LoRA Adapters")

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.visual`: `get_input_embeddings` not auto‑handled for Qwen3_5VisionModel; please override in the subclass.. Falling back to pre-forward hook.


Added LoRA Adapters


In [6]:
tokenizer.video_processor.do_sample_frames=False

In [12]:
total_sample = len(train_ds)
print(f"Total Samples: {total_sample}")

Total Samples: 468901


In [71]:
i = 15

In [133]:
sample_data = train_ds[i]
print(f" Dataset: {sample_data['dataset']}")
i += 1

 Dataset: robofac


In [134]:
# Test on inference
FastVisionModel.for_inference(model)
input_text = tokenizer.apply_chat_template(
    [sample_data["messages"][0]],
    add_generation_prompt = True,
    tokenize = False,
    return_dict = True
)
print("-"*20)
print("Input Text:")
print(input_text)
print("Input Text Ended")
print("-"*20)

inputs = tokenizer.apply_chat_template(
    [sample_data["messages"][0]],
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to(model.device)

--------------------
Input Text:
<|im_start|>user
<|vision_start|><|video_pad|><|vision_end|>You are a robot manipulation assistant. Answer questions about robot tasks, object affordances, spatial relationships, and manipulation strategies based on the provided image or video frame.

This is a video of a robotic arm performing a task, during which subtask did the error happen? (Your answer should choose one of the following options: ['Grasp the cup.', 'Pull the screwdriver off the table.', 'Grasp the lock.', 'Rotate the box to an upright position.', 'Reach for the mouse.'])<|im_end|>
<|im_start|>assistant
<think>

</think>


Input Text Ended
--------------------


In [135]:
text_streamer = TextStreamer(tokenizer,skip_prompt = True)

In [136]:
outputs = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    temperature = 1.5,
    min_p = 0.1,
    use_cache = True
)

Grasp the cup.<|im_end|>
<|endoftext|>


In [137]:
print(f"Ground Truth: {sample_data["messages"][1]["content"]}")

Ground Truth: Grasp the cup.


# Start Training 

In [138]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [139]:
os.environ["WANDB_PROJECT"] = "Final Year Project"  # name your W&B project
os.environ["WANDB_LOG_MODEL"] = "checkpoint"

In [141]:
FastVisionModel.for_training(model) # Enable for training!

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    data_collator = UnslothVisionDataCollator(model, tokenizer), # Must use!
    train_dataset = train_ds,
    args = SFTConfig(
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 8,
        warmup_steps = 100,
        # max_steps = 75000,
        num_train_epochs = 1, # Set this instead of max_steps for full training runs
        learning_rate = 1e-5,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs/stage_1",
        report_to = "wandb",     # For Weights and Biases

        # You MUST put the below items for vision finetuning:
        remove_unused_columns = False,
        dataset_text_field = "",
        dataset_kwargs = {"skip_prepare_dataset": True},
        max_length = 2048,
    ),
)

Unsloth: Model does not have a default image size - using 512


In [142]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA RTX A4000. Max memory = 15.992 GB.
9.27 GB of memory reserved.


In [ ]:
trainer_stats = trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 468,901 | Num Epochs = 1 | Total steps = 58,613
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 8 x 1) = 8
 "-____-"     Trainable parameters = 38,756,352 of 4,578,021,888 (0.85% trained)
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/ubuntu/.netrc.
wandb: Currently logged in as: shreethar2004 (shreethar2004-universiti-teknikal-malaysia) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.100014
2,3.008157
3,2.748574
4,3.153054
5,3.173517
6,2.793581
7,3.840307
8,2.685029
9,2.645264
10,3.218878


/home/ubuntu/Shree_FYP/.venv/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
wandb: Adding directory to artifact (outputs/stage_1/checkpoint-500)... Done. 0.4s
/home/ubuntu/Shree_FYP/.venv/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
wandb: Adding directory to artifact (outputs/stage_1/checkpoint-1000)... Done. 0.4s
wandb: Adding directory to artifact (outputs/stage_1/checkpoint-1500)... Done. 0.4s
/home/ubuntu/Shree_FYP/.venv/lib/python3.12/site-packages/PIL/Image.py:1137: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
wandb: Adding directory to artifact (outputs/stage_1/checkpoint-2000)... Done. 0.4s
/home/ubuntu/Shree_FYP/.venv/lib/python3.12/site-packages/PIL/Image.py:1137: UserWa

In [1]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

NameError: name 'torch' is not defined

# Completed Training Stage 1

In [ ]:
iterator = iter(train_data)

In [ ]:
next_sample = next(iterator)

In [ ]:
data = next_sample

In [ ]:
data['dataset']

In [ ]:
data_message = data['messages']

In [ ]:
data_message

In [ ]:
# Test on inference
FastVisionModel.for_inference(model)
input_text = tokenizer.apply_chat_template(
    [data_message[0]],
    add_generation_prompt = True,
    tokenize = False,
    return_dict = True
)
print("-"*20)
print("Input Text:")
print(input_text)
print("Input Text Ended")
print("-"*20)

inputs = tokenizer.apply_chat_template(
    [data_message[0]],
    add_generation_prompt = True,
    tokenize = True,
    return_dict = True,
    return_tensors = "pt",
).to(model.device)

In [ ]:
text_streamer = TextStreamer(tokenizer,skip_prompt = True)

In [ ]:
outputs = model.generate(
    **inputs,
    streamer = text_streamer,
    max_new_tokens = 256,
    temperature = 1.5,
    min_p = 0.1,
    use_cache = True
)